# 10. Production


## Learning Objectives

Learn how to test, deploy, and monitor agents.

This notebook covers:
- Local development and debugging with LangSmith Studio
- Deterministic agent testing with `GenericFakeChatModel`
- Trajectory-based tests for validating tool call order
- Web-based interaction with Agent Chat UI
- Deployment with LangGraph Platform or your own server
- Observability with LangSmith


## 10.1 Environment Setup


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

from langchain.agents import create_agent
from langchain.tools import tool

print("환경 준비 완료.")

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 10.2 LangSmith Studio

Develop and debug agents locally.

To use Studio, you need:
- a `langgraph.json` config file
- a local server started with `langgraph dev`
- interactive testing through the Studio UI

Studio is a powerful tool for visualizing agent execution flow and debugging each step.


In [ ]:
# langgraph.json 설정 예시
import json

langgraph_config = {
    "dependencies": ["."],
    "graphs": {
        "agent": "./agent.py:agent"
    },
    "env": ".env"
}

print("langgraph.json 설정 예시:")
print(json.dumps(langgraph_config, indent=2))
print("\n실행 방법:")
print("  $ langgraph dev")
print("  → http://localhost:2024 에서 Studio UI 접근")

## 10.3 Agent Testing

With `GenericFakeChatModel`, you can test an agent deterministically without making real API calls.

Benefits of this approach:
- No API cost during tests
- Always returns the same result, which is ideal for CI/CD pipelines
- Lets you validate the agent's logic independently (tool calls, branching, and so on)


In [ ]:
from langchain_core.language_models import GenericFakeChatModel
from langchain.messages import AIMessage
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_capital(country: str) -> str:
    """국가의 수도를 반환합니다."""
    capitals = {"Korea": "Seoul", "Japan": "Tokyo", "France": "Paris"}
    return capitals.get(country, "알 수 없음")

# 가짜 모델로 결정론적 테스트
fake_model = GenericFakeChatModel(
    messages=iter([
        AIMessage(content="대한민국의 수도는 서울입니다.")
    ])
)

# 테스트 에이전트
test_agent = create_agent(
    model=fake_model,
    tools=[get_capital],
    system_prompt="당신은 지리 전문가입니다.",
)

print("GenericFakeChatModel 테스트:")
print("  → 결정론적 응답으로 에이전트 동작을 테스트합니다")
print("  → CI/CD 파이프라인에서 API 호출 없이 테스트 가능")

## 10.4 Trajectory-Based Testing

Validate the order in which the agent calls tools. A trajectory test checks whether the agent uses tools in the expected order and whether the final response matches your expectation.


In [ ]:
# 트라젝토리 테스트 예시
def test_agent_trajectory():
    """에이전트가 예상된 순서로 도구를 호출하는지 테스트합니다."""
    result = test_agent.invoke(
        {"messages": [{"role": "user", "content": "대한민국의 수도는 어디인가요?"}]}
    )
    
    messages = result["messages"]
    
    # 검증: 메시지가 존재하는지
    assert len(messages) > 0, "에이전트가 응답하지 않았습니다"
    
    # 검증: 마지막 메시지가 AI 응답인지
    last_msg = messages[-1]
    assert hasattr(last_msg, 'content'), "마지막 메시지에 content가 없습니다"
    
    print("✓ 트라젝토리 테스트 통과")
    print(f"  메시지 수: {len(messages)}")
    print(f"  최종 응답: {last_msg.content[:100]}")

try:
    test_agent_trajectory()
except Exception as e:
    print(f"테스트 참고: {e}")

## 10.5 Agent Chat UI

This is a web UI for talking to your agent. It connects to a LangGraph server so you can test the agent directly in the browser.

Key features:
- Real-time streaming chat
- Tool call visualization
- Conversation branching
- Human-in-the-loop approval


In [ ]:
print("Agent Chat UI 설정:")
print("=" * 50)
print("""
# 1. Agent Chat UI 설치
$ npx @anthropic-ai/agent-chat-ui

# 2. LangGraph 서버 시작
$ langgraph dev

# 3. UI에서 http://localhost:2024 연결
#    → 웹 브라우저에서 에이전트와 대화
""")
print("주요 기능:")
print("  - 실시간 스트리밍 채팅")
print("  - 도구 호출 시각화")
print("  - 대화 분기(branching)")
print("  - Human-in-the-loop 승인")

## 10.6 Deployment

You can deploy an agent through LangGraph Platform (managed) or your own server. Choose the option that best fits your production environment.


In [ ]:
print("배포 옵션:")
print("=" * 50)

print("""
# 옵션 1: LangGraph Platform (관리형)
$ langgraph deploy


# 옵션 2: 자체 Docker 배포
$ langgraph build -t my-agent
$ docker run -p 2024:2024 my-agent


# 옵션 3: FastAPI/Flask 래핑
from fastapi import FastAPI

app = FastAPI()


@app.post("/chat")
async def chat(message: str):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        }
    )

    return {
        "response": result["messages"][-1].content
    }
""")

## 10.7 Observability

Use LangSmith to trace agent behavior. When tracing is enabled, every step of agent execution is recorded and can be analyzed.

LangSmith lets you inspect:
- The complete execution flow of each agent call
- Model input/output, tool calls, and token usage
- Latency, errors, and cost tracking


In [ ]:
print("LangSmith 관측성 설정:")
print("=" * 50)
print("""
# .env에 설정:
LANGSMITH_API_KEY=lsv2-...
LANGSMITH_TRACING=true

# 설정 후 에이전트 실행 시 자동으로 트레이스가 기록됩니다.
# https://smith.langchain.com 에서 확인:
#   - 각 에이전트 호출의 전체 실행 흐름
#   - 모델 입/출력, 도구 호출, 토큰 사용량
#   - 지연 시간, 에러, 비용 추적
""")

# 트레이싱 상태 확인
import os
tracing = os.environ.get("LANGSMITH_TRACING", "false")
api_key = os.environ.get("LANGSMITH_API_KEY", "")
print(f"\n현재 트레이싱 상태: {'활성' if tracing.lower() == 'true' else '비활성'}")
print(f"API 키 설정: {'✓' if api_key else '✗ 미설정'}")

## 10.8 Production Checklist

Before deploying an agent to production, review the following checklist.

| Item | Tool | Status |
|------|------|------|
| Unit tests | `GenericFakeChatModel`, `pytest` | |
| Trajectory tests | Custom validation functions | |
| Observability | LangSmith tracing | |
| Error handling | `try/except`, retry logic | |
| Security | API key management, input validation, guardrails | |
| Deployment environment | Docker, LangGraph Platform | |
| Monitoring | LangSmith dashboards, alert configuration | |
| Documentation | API docs, agent behavior notes | |


## 10.9 Summary

This notebook covered:

| Topic | Key Idea |
|------|----------|
| **LangSmith Studio** | Use `langgraph dev` to debug agents visually on your local machine |
| **Agent testing** | Run deterministic tests with `GenericFakeChatModel` and no API calls |
| **Trajectory tests** | Validate tool call order and final responses |
| **Agent Chat UI** | Talk to agents in the browser and visualize tool usage |
| **Deployment** | Deploy with LangGraph Platform, Docker, FastAPI, and related options |
| **Observability** | Use LangSmith to track execution flow, token usage, and cost |

This completes the LangChain v1 agent track. You covered the full lifecycle of agent development, from basic concepts to production deployment.

### Next Steps
→ Continue to **[11_mcp.ipynb](./11_mcp.ipynb)**
→ Or jump to the **[LangGraph intermediate track](../03_langgraph/01_introduction.ipynb)**
